# Evaluation of the baseline detector

This notebook evaluates the single-class detector from `01_training` (the weights in `inference.weights` in the config) on the validation split.

**Caveat:** there is no separate test split yet. The validation set was already used to select `best.pt` during training, and it is also used here to choose the confidence threshold. The numbers in this notebook are therefore optimistic. The validation set is also small, so a single image can shift the results noticeably. A proper test split should be added once the dataset grows.

Two kinds of evaluation are used:
- **mAP** from Ultralytics' `val()`, which measures the quality of the model's weights across all confidence thresholds.
- **Per-image counts** of found, false and missed boxes, computed through `Detector` from the `object_detection` package, which is the same code path the API uses.

In [ ]:
# Importing libraries and defining paths

from ultralytics import YOLO
from IPython.display import display
from PIL import Image, ImageDraw

from object_detection.config.loader import PROJECT_ROOT, load_config
from object_detection.detection.evaluation import iou, load_true_boxes, match_detections
from object_detection.detection.inference import Detector
from object_detection.utils.visualization import draw_detections

cfg = load_config()
VALID = cfg.data.detector_yaml.parent / "valid"
val_imgs = sorted((VALID / "images").glob("*"))

print(cfg.inference.weights)
print(f"{VALID} - {len(val_imgs)} images")

In [ ]:
# Evaluating overall quality (mAP on the validation set)

model = YOLO(cfg.inference.weights)

metrics = model.val(
    data=str(cfg.data.detector_yaml),
    split="val",
    project=str(PROJECT_ROOT / "runs"),
    # Named after the run being evaluated: runs/<run>/weights/best.pt -> "<run>-val"
    name=f"{cfg.inference.weights.parents[1].name}-val",
    exist_ok = True,
)

print(f"mAP50:  {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision:  {metrics.box.mp:.3f}")
print(f"Recall:  {metrics.box.mr:.3f}")

The detector reaches **mAP50 0.973** and **mAP50-95 0.903** on the validation set, matching the best epoch during training, which confirms the correct weights and dataset were evaluated. The high mAP50-95 means the boxes are not only in the right place but also tightly fitted.

Precision (0.935) and recall (0.951) above are reported by Ultralytics at the confidence where F1 peaks (around 0.5), not at the confidence threshold currently in the config (0.25).

The confusion matrix saved in `runs/baseline-val/` counts only predictions above confidence 0.25, with a match requiring IoU ≥ 0.45:

|                      | Actually an object | Actually background |
|----------------------|--------------------|---------------------|
| **Predicted object** | 43 found           | 7 false boxes       |
| **Predicted background** | 2 missed       | –                   |

At 0.25 this gives a precision of 43/50 = 0.86, noticeably lower than the 0.935 at the F1 peak, while recall barely changes. This suggests the current confidence threshold is too low, which will be tested properly below.


In [ ]:
# Checking the ground-truth loader

for box in load_true_boxes(val_imgs[0]):
    print(f"{[round(v) for v in box]}")

total_boxes = 0
background_imgs = 0

for img_path in val_imgs:
    true_boxes = load_true_boxes(img_path)
    total_boxes += len(true_boxes)
    if not true_boxes:
        background_imgs += 1

print(f"Total Boxes: {total_boxes}, Background Images: {background_imgs}")

In [ ]:
# Checking the IoU function

test_cases = [
    ((0, 0, 10, 10), (0, 0, 10, 10), 1.0),     # identical
    ((0, 0, 10, 10), (20, 20, 30, 30), 0.0),   # far apart
    ((0, 0, 10, 10), (5, 0, 15, 10), 0.333),   # overlap 50, union 150
    ((0, 0, 10, 10), (10, 0, 20, 10), 0.0),    # edges touch, no shared area
]

for box_a, box_b, expected in test_cases:
    result = iou(box_a, box_b)
    print(f"{box_a} vs {box_b}: got {result:.3f}, expected {expected:.3f}")

In [ ]:
# Checking IoU on real predictions

# Pinned to 0.25, the threshold these checks were written at; the markdown below describes results at 0.25.
detector = Detector(cfg.inference.weights, 0.25)
detections = detector.predict(val_imgs[0])

for true_box in load_true_boxes(val_imgs[0]):
    best_iou = max(iou(true_box, detection.bbox) for detection in detections)
    print(f"{[round(v) for v in true_box]} Best IoU: {best_iou:.3f}")

In [ ]:
# Checking the matching function

found, false_boxes, missed = match_detections(detections, load_true_boxes(val_imgs[0]))
print(f"First Image: found {len(found)}, false {len(false_boxes)}, missed {len(missed)}")

total_found = 0
total_false = 0
total_missed = 0
for img_path in val_imgs:
    found, false_boxes, missed = match_detections(detector.predict(img_path), load_true_boxes(img_path))
    total_found += len(found)
    total_false += len(false_boxes)
    total_missed += len(missed)

print(f"All Images: found {total_found}, false {total_false}, missed {total_missed}")

Matching predictions to the labelled boxes (one-to-one, highest confidence first, IoU ≥ 0.5) at confidence 0.25 gives **43 found, 12 false boxes and 2 missed**. Every labelled box is accounted for (43 + 2 = 45).

This is 12 false boxes where Ultralytics' confusion matrix reported 7. The difference is not a bug in the matching: `val()` and `Detector.predict()` prepare images differently (batching, resizing and padding, and half precision on the GPU), which shifts the confidences slightly. With the same weights, `val()` produced 50 boxes above 0.25, while `Detector.predict()` produced 55.

Since the API will call `Detector.predict()`, the counts from this notebook's own matching are the ones that describe the service, and they are what the confidence threshold will be chosen on.


In [ ]:
# Choosing the confidence threshold

CANDIDATE_CONFS = [round(0.1 + 0.05 * i, 2) for i in range(17)]

low_conf_detector = Detector(cfg.inference.weights, conf=min(CANDIDATE_CONFS))
results = {
    img_path: (low_conf_detector.predict(img_path), load_true_boxes(img_path))
    for img_path in val_imgs
}

for conf in CANDIDATE_CONFS:
    total_found = 0
    total_false = 0
    total_missed = 0
    background_false = 0

    for detections, true_boxes in results.values():
        kept = [detection for detection in detections if detection.confidence >= conf]
        found, false_boxes, missed = match_detections(kept, true_boxes)
        total_found += len(found)
        total_false += len(false_boxes)
        total_missed += len(missed)
        if not true_boxes:
            background_false += len(false_boxes)

    detected = total_found + total_false
    precision = total_found / detected if detected else 0.0
    recall = total_found / (total_found + total_missed)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    print(f"{conf:>5} {total_found:>5} {total_false:>5} {total_missed:>5} {background_false:>5} {precision:>6.3f} {recall:>6.3f} {f1:>6.3f}")

For this project, a missed object is a more costly mistake than a falsely-detected object/box, so we will prioritise the recall of the model. Therefore, 0.5 is a good balance between precision and recall, and it drops false positives from 7 to 5 from a conf of 0.45, while keeping the number missed objects reasonable as opposed to higher confs.

In [ ]:
# Error analysis at the confidence threshold of 0.5

ANALYSIS_CONF = 0.5

for img_path, (detections, true_boxes) in results.items():
    kept = [detection for detection in detections if detection.confidence >= ANALYSIS_CONF]
    found, false_boxes, missed = match_detections(kept, true_boxes)

    if not false_boxes and not missed:
        continue

    print(f"{img_path.name}: found {len(found)}, false {len(false_boxes)}, missed {len(missed)}")

    for detection in false_boxes:
        best_iou = max((iou(detection.bbox, true_box) for true_box in true_boxes), default=0.0)
        print(f"    false box, conf {detection.confidence:.3f}, best IoU with a labelled box {best_iou:.2f}")

    with Image.open(img_path) as image:
        annotated = draw_detections(image, false_boxes)

    draw = ImageDraw.Draw(annotated)
    line_width = max(2, round(max(annotated.size) / 400))
    for true_box in true_boxes:
        draw.rectangle(true_box, outline=(0, 200, 0), width=line_width)
    for true_box in missed:
        draw.rectangle(true_box, outline=(255, 200, 0), width=line_width * 2)

    annotated.thumbnail((800, 800))
    display(annotated)

## Decision: confidence threshold

**Chosen `conf`: 0.5** (previously 0.25, Ultralytics' default).

At 0.25 the detector finds 43 of 45 objects but draws 12 false boxes. At 0.5 it finds 41, with 5 false boxes and 4 missed.

The threshold was not set at the F1 peak (0.75, with zero false boxes) for three reasons:
- **Missed objects cannot be recovered later.** A false box still reaches Stage 2, where a low match score can reject it; a missed object never does. The detector should lean towards recall.
- **Most false boxes at 0.5 are not junk.** Of the 5: one is a duplicate, two span overlapping objects, one is at the edge of an image and may be a real everyday object that is not labelled, and only one (the chair handle) is clearly background. Raising the threshold would hide these rather than fix them.
- **0.75 sits right before a cliff**, where 3 more real objects are lost at 0.8, and on this small validation set the differences between 0.5 and 0.75 come down to a handful of boxes.

**Remaining errors point to data, not the threshold:**
- **Overlapping objects** (06-18-57) are merged into boxes spanning several objects, causing both false boxes and misses. More training images with touching or overlapping objects are needed.
- **Unusual poses** such as a bag leaning against a wall (06-18-58) get low confidence. More varied poses are needed.
- **Labelling consistency:** whether straps and cables belong inside the box should follow one rule, and it should be decided whether everyday objects outside the 8 types are labelled too.

As noted at the top, this threshold was chosen on the validation set and should be re-checked on a test split once the dataset grows.
